# Cut Athena query cost by 90 percent with partitioning, Parquet, and workgroup limits

The analytics team just sent a Slack message: their Athena bill for last month was $480, almost all of it from one query running five times a day on the raw CSV table. You have one session to convert the table to partitioned Parquet, configure partition projection, and add a workgroup guardrail so the next runaway query gets killed at 1 GB instead of $1 per run.

The literal scenario from the cert YAML is 200 GB of CSV per query. This lab uses a small seeded dataset (about 10 MB of CSV and 2 MB of Parquet) so you can prove the optimization pattern via `DataScannedInBytes` deltas without literally scanning 200 GB. The 80-percent-plus reduction holds on both scales; what matters here is that you observe the delta and learn the workgroup-cap pattern.

The four deliverables map to four checkpoints:
1. An Athena workgroup with `BytesScannedCutoffPerQuery` set to 1 GB or less, output location wired to the lab bucket, and `State=ENABLED`. This is the safety net; it gates the rest of the lab.
2. A Glue table `events_csv` over `raw-csv/` that runs a baseline query in the lab workgroup and reports a non-zero `DataScannedInBytes`. The byte count is cached in `CSV_SCAN_BYTES` for the final comparison.
3. A Glue table `events_parquet` over `curated-parquet/` with year and month partition keys plus partition projection enabled (`projection.enabled=true`, integer range projections on year and month, and a `storage.location.template` that resolves `${year}` and `${month}`).
4. The same logical query against `events_parquet` scans at least 80 percent less data than the CSV baseline.

**Time.** About 55 minutes hands-on. The seed step and Athena polls dominate wall-clock; the writing is short.

**Cost.** This lab costs under 5 cents in the happy path. Athena bills at $5 per TB scanned and you are scanning megabytes; even on the unoptimized CSV table you would have to run the query 10,000 times to spend a dollar. The workgroup cap you build in Checkpoint 1 guarantees a runaway query cannot blow that budget. The cleanup cell at the end deletes the workgroup, the tables, the database, and the bucket so the bill stops the moment you finish.

This lab maps to AWS DEA-C01 Domain 2: Data Store Management (26% of exam weight).


In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.7.0


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT
# section 12.

import atexit
import csv
import getpass
import io
import json
import random
import re
import time
from datetime import datetime, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "athena-query-optimization"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[4].slug exactly
REGION = "us-east-1"  # all DEA-C01 labs run in us-east-1 per LAB-CREATION-BLUEPRINT section 15

# Resource names. Bucket name appends the account ID for global uniqueness.
DATABASE_NAME = f"labrun_{LAB_ID.replace('-', '_')}_db"
WORKGROUP_NAME = f"labrun-{LAB_ID}-wg"
CSV_TABLE_NAME = "events_csv"
PARQUET_TABLE_NAME = "events_parquet"
BUCKET_NAME = None  # resolved after STS once the account ID is known

# Per-query data-scan cap on the lab workgroup. 1 GiB in bytes. Checkpoint 1
# requires this value or lower; the lab dataset is megabytes so any real query
# is comfortably under the cap.
BYTES_CUTOFF_PER_QUERY = 1_073_741_824

# Seed config. Deterministic so the row counts and scan-byte deltas are stable
# across student runs.
SEED = 42
ROWS_PER_MONTH = 5_000
MONTHS = [(2024, 11), (2024, 12), (2025, 1), (2025, 2)]

# Module-level cache for Checkpoint 4. The Task 2 cell runs the baseline CSV
# query and writes the byte count here so Checkpoint 4 can compare without
# rerunning the expensive scan.
CSV_SCAN_BYTES: int | None = None
PARQUET_SCAN_BYTES: int | None = None


In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"All DEA-C01 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast with a
# clear message rather than waiting for the first Athena call to error out.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve the bucket name now that we know the account ID. S3 bucket names
# must be globally unique.
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# The manifest is module-level and in reverse-creation order. Lab 05 has no
# critical (hourly-billed) resources. Athena bills per query; idle workgroups
# are free. The workgroup cap built in Checkpoint 1 is what keeps a runaway
# baseline query from accidentally costing real money.
#
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution if any
# tagged resources from a prior session exist (not just print a warning).
#
# Note: labrun-checks 0.3.0 ships AWS adapters for s3_bucket, iam_role,
# glue_database, and glue_table. It does NOT yet support athena_workgroup. A
# _LabAdapter subclass extending AwsCleanupAdapter is defined in the cleanup
# cell at the bottom of this notebook and passed to run_cleanup. The new
# resource type is still declared declaratively here so the canonical
# summary, atexit handler, and orphan scan all see it.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="athena_workgroup",
        id=WORKGROUP_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws athena delete-work-group "
            f"--work-group {WORKGROUP_NAME} --recursive-delete-option"
        ),
    ),
    CleanupResource(
        type="glue_table",
        id=PARQUET_TABLE_NAME,
        region=REGION,
        parent=DATABASE_NAME,
        cli_delete_command=(
            f"aws glue delete-table --database-name {DATABASE_NAME} "
            f"--name {PARQUET_TABLE_NAME}"
        ),
    ),
    CleanupResource(
        type="glue_table",
        id=CSV_TABLE_NAME,
        region=REGION,
        parent=DATABASE_NAME,
        cli_delete_command=(
            f"aws glue delete-table --database-name {DATABASE_NAME} "
            f"--name {CSV_TABLE_NAME}"
        ),
    ),
    CleanupResource(
        type="glue_database",
        id=DATABASE_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-database --name {DATABASE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell is the authoritative entry point; this is the
    safety net for kernel crashes. Errors are swallowed because atexit
    handlers must not raise.
    """
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter()) if "_LabAdapter" in globals() else run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for leftover
    state with the lab's tag and surface the cleanup command before creating
    any new resources.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources or")
    print("collide on the workgroup and table names. Run the cleanup cell at")
    print("the bottom of this notebook first, or remove them manually with")
    print("the AWS CLI commands the cleanup cell prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Task 1: Stand up the lab workgroup with a per-query data-scan cap

Before you touch the data, build the safety net. Athena pricing is $5 per TB scanned and the literal scenario at the top of this lab describes a 200 GB CSV table that costs $1 per query. A student who forgets to filter or who queries the wrong table can run up a real bill quickly. The workgroup-level `BytesScannedCutoffPerQuery` is how production teams keep that from happening: any query in the workgroup that crosses the cutoff is cancelled before it runs to completion.

Three things to configure on `labrun-athena-query-optimization-wg`:

1. **`BytesScannedCutoffPerQuery=1073741824`** (exactly 1 GiB). The lab dataset is megabytes, so legitimate queries clear this comfortably; a misconfigured query that goes hot stops before it spends money.
2. **`ResultConfiguration.OutputLocation=s3://{BUCKET}/athena-results/`**. Athena requires a per-query result-set landing zone. Wiring it to the lab bucket means cleanup tears it down with everything else.
3. **`State=ENABLED`**. New workgroups default to enabled; this rule is here so a student who used `athena.update_work_group` on an existing disabled workgroup catches the gap.

The S3 bucket itself also gets created in this task, because Athena will refuse to set the output location to a bucket that does not exist. Order in this cell: create bucket, tag bucket, create workgroup with the three settings above.

Checkpoint 1 will reject any value above 1 GiB and any missing output location. The rest of the lab depends on the cap being in place per RESOURCE-SAFETY-SPEC Section 3 Lab 5 mitigation.


In [ ]:
# NBVAL_SKIP
# Task 1: Create the lab bucket and the Athena workgroup with a 1 GiB
# per-query data-scan cap. The workgroup is the safety net; the rest of the
# lab depends on the cap being in place.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Create the bucket. us-east-1 rejects LocationConstraint; other regions
# require it. All DEA-C01 labs run in us-east-1, so the simple call works.
# YOUR CODE: Create the S3 bucket with s3.create_bucket(Bucket=BUCKET_NAME).

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

athena_results_uri = f"s3://{BUCKET_NAME}/athena-results/"

# YOUR CODE: Create the Athena workgroup with athena.create_work_group(
#   Name=WORKGROUP_NAME,
#   Configuration={
#       "ResultConfiguration": {"OutputLocation": athena_results_uri},
#       "BytesScannedCutoffPerQuery": BYTES_CUTOFF_PER_QUERY,
#       "EnforceWorkGroupConfiguration": True,
#       "PublishCloudWatchMetricsEnabled": False,
#   },
#   Description=f"Lab workgroup for {LAB_ID} with a 1 GiB per-query scan cap.",
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
#
# If the workgroup already exists from a prior session inside the same notebook
# run, wrap the call in try/except ClientError and on InvalidRequestException
# (the code Athena returns for an existing workgroup), call
# athena.update_work_group(
#   WorkGroup=WORKGROUP_NAME,
#   ConfigurationUpdates={
#       "ResultConfigurationUpdates": {"OutputLocation": athena_results_uri},
#       "BytesScannedCutoffPerQuery": BYTES_CUTOFF_PER_QUERY,
#       "EnforceWorkGroupConfiguration": True,
#   },
#   State="ENABLED",
# ).

print(f"Workgroup created: {WORKGROUP_NAME}")
print(f"  OutputLocation: {athena_results_uri}")
print(f"  BytesScannedCutoffPerQuery: {BYTES_CUTOFF_PER_QUERY} bytes (1 GiB)")
print(f"  State: ENABLED")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Athena workgroup exists, BytesScannedCutoffPerQuery is set
# and at or below 1 GiB, OutputLocation points at the lab bucket's
# athena-results/ prefix, and State is ENABLED.

CAP_LIMIT = 1_073_741_824  # 1 GiB
EXPECTED_OUTPUT_LOCATION_SUFFIX = "/athena-results/"


def checkpoint_1(session):
    try:
        athena_client = boto3.client(
            "athena",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            wg_resp = athena_client.get_work_group(WorkGroup=WORKGROUP_NAME)
        except ClientError as e:
            code = e.response["Error"]["Code"]
            if code in ("InvalidRequestException", "ResourceNotFoundException"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Athena workgroup {WORKGROUP_NAME!r} does not exist. "
                        f"Run the Task 1 cell to create it."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        wg = wg_resp.get("WorkGroup", {})
        state = wg.get("State", "")
        if state != "ENABLED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Workgroup State is {state!r}, expected 'ENABLED'. Re-enable "
                    f"it with athena.update_work_group(WorkGroup=WORKGROUP_NAME, "
                    f"State='ENABLED')."
                ),
            )

        config = wg.get("Configuration", {})
        cutoff = config.get("BytesScannedCutoffPerQuery")
        if cutoff is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Workgroup has no BytesScannedCutoffPerQuery set. Add a "
                    "BytesScannedCutoffPerQuery of at most 1 GB to the "
                    "workgroup before continuing; the rest of the lab depends "
                    "on the cap being in place."
                ),
            )
        if cutoff > CAP_LIMIT:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"BytesScannedCutoffPerQuery is {cutoff} bytes, expected "
                    f"<= {CAP_LIMIT} (1 GiB). Reduce the cap before continuing; "
                    f"the rest of the lab depends on the cap being in place."
                ),
            )

        result_config = config.get("ResultConfiguration", {})
        output_location = result_config.get("OutputLocation", "")
        expected_prefix = f"s3://{BUCKET_NAME}/athena-results/"
        if output_location != expected_prefix:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"OutputLocation is {output_location!r}, expected "
                    f"{expected_prefix!r}. Athena writes per-query result sets "
                    f"to this prefix; the lab bucket is the right home so "
                    f"cleanup tears it down with everything else."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks four properties of the workgroup in order: existence, `State == "ENABLED"`, `BytesScannedCutoffPerQuery` set and `<= 1 GiB`, `ResultConfiguration.OutputLocation` matches `s3://{BUCKET}/athena-results/`. Read the failure reason. The most common miss on the first run is forgetting `BytesScannedCutoffPerQuery` in the `Configuration` dict at create time; `update_work_group` is the cleanup. The next most common miss is forgetting to create the bucket first; `create_work_group` does not validate the output-location bucket exists, but `start_query_execution` will fail later if it does not.

</details>

<details><summary>Hint 2 (direction)</summary>

Two API calls in this task. `s3.create_bucket(Bucket=BUCKET_NAME)` (no `LocationConstraint` in us-east-1). `athena.create_work_group(Name=WORKGROUP_NAME, Configuration={"ResultConfiguration": {"OutputLocation": athena_results_uri}, "BytesScannedCutoffPerQuery": BYTES_CUTOFF_PER_QUERY, "EnforceWorkGroupConfiguration": True, "PublishCloudWatchMetricsEnabled": False}, Description=..., Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])`. If the workgroup already exists from a prior cell run, catch `ClientError` with code `InvalidRequestException` and call `athena.update_work_group` with `ConfigurationUpdates` carrying the same `BytesScannedCutoffPerQuery` and `ResultConfigurationUpdates.OutputLocation` plus `State="ENABLED"`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
s3.create_bucket(Bucket=BUCKET_NAME)
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

try:
    athena.create_work_group(
        Name=WORKGROUP_NAME,
        Configuration={
            "ResultConfiguration": {"OutputLocation": athena_results_uri},
            "BytesScannedCutoffPerQuery": BYTES_CUTOFF_PER_QUERY,
            "EnforceWorkGroupConfiguration": True,
            "PublishCloudWatchMetricsEnabled": False,
        },
        Description=f"Lab workgroup for {LAB_ID} with a 1 GiB per-query scan cap.",
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except ClientError as e:
    if e.response["Error"]["Code"] == "InvalidRequestException":
        athena.update_work_group(
            WorkGroup=WORKGROUP_NAME,
            ConfigurationUpdates={
                "ResultConfigurationUpdates": {"OutputLocation": athena_results_uri},
                "BytesScannedCutoffPerQuery": BYTES_CUTOFF_PER_QUERY,
                "EnforceWorkGroupConfiguration": True,
            },
            State="ENABLED",
        )
    else:
        raise
```

If `create_work_group` returns `AccessDeniedException`, your `labrun-test` IAM user is missing `AmazonAthenaFullAccess`. If `create_bucket` returns `BucketAlreadyOwnedByYou`, the orphan scan in setup should have caught a leftover from a prior run; that path is treated as an error here on purpose. Run the cleanup cell at the bottom of the notebook to clear state, then re-run setup and Task 1.

</details>


**Wallet check.** Still at $0.00. The bucket itself is free; you only pay for storage and requests. The workgroup is also free (Athena bills per query, not per workgroup). The cap you just put in place is the seatbelt for the rest of the lab.


## Task 2: Seed the bucket with CSV, register `events_csv`, and run the baseline query

Now you need the unoptimized table the analytics team has been paying for. Three pieces in this task:

1. **Seed a deterministic CSV under `raw-csv/events.csv`.** 20,000 rows across November 2024 through February 2025 (5,000 per month). Columns: `event_id`, `event_year`, `event_month`, `event_day`, `customer_id`, `amount_cents`, `event_type`. Plain CSV with a header. About 1 MB on disk. The whole point of the unoptimized table is that Athena has to scan every byte on every query because there is no partition pruning and no columnar format.
2. **Register `events_csv` in the Glue Data Catalog.** Database `labrun_athena_query_optimization_db`, table name `events_csv`, `StorageDescriptor.Location` at `s3://{BUCKET}/raw-csv/`, OpenCSV SerDe, no partition keys. CSV is the format the cert YAML scenario indicts; the table is the baseline.
3. **Run the baseline query in the lab workgroup and cache the byte count.** `SELECT count(*) FROM events_csv WHERE event_year = 2025` against the lab workgroup. The query polls until `SUCCEEDED`, reads `Statistics.DataScannedInBytes` from `get_query_execution`, and writes the value into the module-level `CSV_SCAN_BYTES`. Checkpoint 4 reads that value to compute the 80%-reduction target.

The workgroup cap from Checkpoint 1 is the safety net. The baseline query on a 1 MB CSV will be well under the 1 GiB cutoff; even the literal-scenario 200 GB version would have been cancelled at 1 GiB if the cap were in place. Read that as the lesson: workgroup caps protect against unbounded scans before optimization, not just after.


In [ ]:
# NBVAL_SKIP
# Task 2: Seed the CSV, register the events_csv Glue table, and run the
# baseline query in the lab workgroup. The byte count from the baseline is
# cached in CSV_SCAN_BYTES for Checkpoint 4.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

EVENT_TYPES = ["click", "view", "purchase", "signup", "logout"]


def _generate_events_csv() -> bytes:
    """Build the seed CSV in memory. 20,000 rows total, deterministic."""
    random.seed(SEED)
    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerow([
        "event_id", "event_year", "event_month", "event_day",
        "customer_id", "amount_cents", "event_type",
    ])
    event_id = 0
    for (year, month) in MONTHS:
        # Use 28 as the max day so February works without calendar math.
        for _ in range(ROWS_PER_MONTH):
            event_id += 1
            day = random.randint(1, 28)
            customer = f"cust_{random.randint(1000, 9999)}"
            amount = random.randint(100, 99999)
            etype = random.choice(EVENT_TYPES)
            writer.writerow([
                f"evt_{event_id:07d}", year, month, day,
                customer, amount, etype,
            ])
    return buf.getvalue().encode("utf-8")


csv_bytes = _generate_events_csv()
CSV_KEY = "raw-csv/events.csv"

# YOUR CODE: Upload the CSV to s3://{BUCKET_NAME}/raw-csv/events.csv with
# s3.put_object(Bucket=BUCKET_NAME, Key=CSV_KEY, Body=csv_bytes).

total_rows = ROWS_PER_MONTH * len(MONTHS)
print(f"Seeded CSV at s3://{BUCKET_NAME}/{CSV_KEY} ({total_rows} rows, {len(csv_bytes)} bytes)")

# Glue database. Glue Data Catalog is free; creating the database is a
# control-plane call.
try:
    glue.create_database(
        DatabaseInput={
            "Name": DATABASE_NAME,
            "Description": f"Catalog for {LAB_ID}",
        },
    )
    glue.tag_resource(
        ResourceArn=f"arn:aws:glue:{REGION}:{ACCOUNT_ID}:database/{DATABASE_NAME}",
        TagsToAdd={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
    print(f"Glue database created and tagged: {DATABASE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue database {DATABASE_NAME} already exists, reusing.")
    else:
        raise

# events_csv table. OpenCSV SerDe, no partitions, location is raw-csv/.
csv_table_input = {
    "Name": CSV_TABLE_NAME,
    "TableType": "EXTERNAL_TABLE",
    "Parameters": {
        "classification": "csv",
        "skip.header.line.count": "1",
        "has_encrypted_data": "false",
    },
    "StorageDescriptor": {
        "Columns": [
            {"Name": "event_id", "Type": "string"},
            {"Name": "event_year", "Type": "int"},
            {"Name": "event_month", "Type": "int"},
            {"Name": "event_day", "Type": "int"},
            {"Name": "customer_id", "Type": "string"},
            {"Name": "amount_cents", "Type": "int"},
            {"Name": "event_type", "Type": "string"},
        ],
        "Location": f"s3://{BUCKET_NAME}/raw-csv/",
        "InputFormat": "org.apache.hadoop.mapred.TextInputFormat",
        "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
        "SerdeInfo": {
            "SerializationLibrary": "org.apache.hadoop.hive.serde2.OpenCSVSerde",
            "Parameters": {
                "separatorChar": ",",
                "quoteChar": '"',
            },
        },
    },
    "PartitionKeys": [],
}

# YOUR CODE: Register the CSV table with glue.create_table(
#   DatabaseName=DATABASE_NAME,
#   TableInput=csv_table_input,
# ). Wrap in try/except ClientError and on AlreadyExistsException, reuse.


def run_athena_query(sql: str, workgroup: str) -> dict:
    """Submit a query, poll until terminal, return the get_query_execution dict.

    Raises RuntimeError on FAILED or CANCELLED. Caller reads
    Statistics.DataScannedInBytes from the returned dict.
    """
    start = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": DATABASE_NAME},
        WorkGroup=workgroup,
    )
    qid = start["QueryExecutionId"]
    deadline = time.time() + 120  # 2 minutes
    while time.time() < deadline:
        resp = athena.get_query_execution(QueryExecutionId=qid)
        state = resp["QueryExecution"]["Status"]["State"]
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            if state != "SUCCEEDED":
                reason = resp["QueryExecution"]["Status"].get("StateChangeReason", "(no reason)")
                raise RuntimeError(f"Query {qid} ended {state}: {reason}")
            return resp["QueryExecution"]
        time.sleep(2)
    raise RuntimeError(f"Query {qid} did not finish within 2 minutes.")


BASELINE_SQL = f"SELECT count(*) FROM {CSV_TABLE_NAME} WHERE event_year = 2025"
print(f"Running baseline query in {WORKGROUP_NAME}: {BASELINE_SQL}")
baseline_exec = run_athena_query(BASELINE_SQL, WORKGROUP_NAME)
CSV_SCAN_BYTES = baseline_exec.get("Statistics", {}).get("DataScannedInBytes", 0)
print(f"Baseline DataScannedInBytes: {CSV_SCAN_BYTES} bytes ({CSV_SCAN_BYTES / 1024:.1f} KiB)")
print(f"CSV_SCAN_BYTES cached for Checkpoint 4.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: events_csv Glue table is registered at raw-csv/, the baseline
# query ran in the lab workgroup and reported a non-zero DataScannedInBytes,
# and the value is cached in CSV_SCAN_BYTES for Checkpoint 4.


def checkpoint_2(session):
    global CSV_SCAN_BYTES
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        athena_client = boto3.client(
            "athena",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            table_resp = glue_client.get_table(
                DatabaseName=DATABASE_NAME, Name=CSV_TABLE_NAME,
            )
        except ClientError as e:
            if e.response["Error"]["Code"] == "EntityNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Glue table {DATABASE_NAME}.{CSV_TABLE_NAME} does not "
                        f"exist. Run the Task 2 cell to register it."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        table = table_resp["Table"]
        sd = table.get("StorageDescriptor", {})
        expected_location = f"s3://{BUCKET_NAME}/raw-csv/"
        actual_location = sd.get("Location", "")
        if actual_location.rstrip("/") + "/" != expected_location:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"events_csv StorageDescriptor.Location is "
                    f"{actual_location!r}, expected {expected_location!r}. "
                    f"The unoptimized table must point at raw-csv/ so the "
                    f"baseline query scans the full CSV."
                ),
            )

        # Re-run the baseline if not cached. Validator triggers it itself to
        # tolerate students who skipped the task-cell run.
        if CSV_SCAN_BYTES is None or CSV_SCAN_BYTES == 0:
            sql = f"SELECT count(*) FROM {CSV_TABLE_NAME} WHERE event_year = 2025"
            try:
                start = athena_client.start_query_execution(
                    QueryString=sql,
                    QueryExecutionContext={"Database": DATABASE_NAME},
                    WorkGroup=WORKGROUP_NAME,
                )
            except ClientError as e:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Could not start baseline query in {WORKGROUP_NAME!r}: "
                        f"{e}. Check that the workgroup exists and is ENABLED "
                        f"(Checkpoint 1)."
                    ),
                )
            qid = start["QueryExecutionId"]
            deadline = time.time() + 120
            scanned = None
            while time.time() < deadline:
                resp = athena_client.get_query_execution(QueryExecutionId=qid)
                state = resp["QueryExecution"]["Status"]["State"]
                if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
                    if state != "SUCCEEDED":
                        reason = resp["QueryExecution"]["Status"].get("StateChangeReason", "(no reason)")
                        return CheckpointResult(
                            status="fail",
                            error_reason=(
                                f"Baseline query ended {state}: {reason}. The "
                                f"events_csv table is registered but the query "
                                f"could not complete; inspect the table "
                                f"definition and re-run the Task 2 cell."
                            ),
                        )
                    scanned = resp["QueryExecution"].get("Statistics", {}).get("DataScannedInBytes", 0)
                    break
                time.sleep(2)
            if scanned is None:
                return CheckpointResult(
                    status="fail",
                    error_reason="Baseline query did not finish within 2 minutes.",
                )
            CSV_SCAN_BYTES = scanned

        if not CSV_SCAN_BYTES or CSV_SCAN_BYTES <= 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Baseline DataScannedInBytes is {CSV_SCAN_BYTES}, expected > 0. "
                    f"The CSV table is registered but the query scanned zero bytes; "
                    f"confirm s3://{BUCKET_NAME}/raw-csv/events.csv exists."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks three things: `events_csv` exists in the Glue Data Catalog, its `StorageDescriptor.Location` is exactly `s3://{BUCKET}/raw-csv/`, and a baseline `SELECT count(*) ... WHERE event_year = 2025` query in the lab workgroup reports a non-zero `DataScannedInBytes`. Read the failure reason. The most common miss is `s3.put_object` landing the CSV under a different prefix (for example `raw/` instead of `raw-csv/`) so the location matches but Athena scans zero bytes. The next most common miss is forgetting to wrap `glue.create_table` in `try / except` for re-runs in the same session.

</details>

<details><summary>Hint 2 (direction)</summary>

Two API calls in this task. `s3.put_object(Bucket=BUCKET_NAME, Key=CSV_KEY, Body=csv_bytes)` lands the CSV. `glue.create_table(DatabaseName=DATABASE_NAME, TableInput=csv_table_input)` registers the table (wrap in `try / except ClientError` and reuse on `AlreadyExistsException`). The baseline query call (`run_athena_query`) is already written in the cell and writes the byte count into `CSV_SCAN_BYTES`; just let it run. If the baseline reports zero bytes scanned, the table is registered but the data is missing under `raw-csv/`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
s3.put_object(Bucket=BUCKET_NAME, Key=CSV_KEY, Body=csv_bytes)

try:
    glue.create_table(
        DatabaseName=DATABASE_NAME,
        TableInput=csv_table_input,
    )
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        pass
    else:
        raise
```

If the baseline query fails with `HIVE_BAD_DATA: Error parsing field value`, the OpenCSV SerDe is reading the header row as data; the table input already sets `skip.header.line.count=1` in `Parameters`, but a custom rewrite that dropped it would surface here. If the query fails with `Permission denied on S3 path`, your `labrun-test` IAM user is missing `s3:GetObject` on the lab bucket; `AmazonS3FullAccess` covers it. If the byte count comes back as zero, the `s3.put_object` landed under the wrong key; confirm `raw-csv/events.csv` exists.

</details>


**Wallet check.** Still essentially $0.00. Glue Data Catalog is free at this scale. The baseline Athena query scanned about 1 MB, which is a fraction of a penny (Athena bills $5 per TB scanned; 1 MB is 0.0000005 TB, or about $0.0000025). Even if you re-ran the baseline ten times during debugging, you are still under one cent. On the real 200 GB CSV table from the cert scenario, each unoptimized query would have been about $1.


## Task 3: Convert to partitioned Parquet and register `events_parquet` with projection

The CSV table is the before-photo. Now build the after-photo: the same logical data in Parquet, partitioned by `year` and `month`, registered in the Glue Data Catalog with partition projection enabled so Athena does not need a manual `MSCK REPAIR TABLE` or a crawler to discover partitions.

Three pieces in this task:

1. **Convert the seeded CSV to partitioned Parquet under `curated-parquet/year=YYYY/month=MM/`.** The cell uses `pyarrow` (preinstalled in Colab and `awswrangler` is not needed) to read the in-memory CSV and write one Parquet file per `(year, month)` partition. The brief notes the lab ships the conversion as a helper so students stay focused on the catalog config rather than rewriting Spark; the helper is in the cell as `_convert_csv_to_partitioned_parquet`.
2. **Register `events_parquet` in the Glue Data Catalog.** Database is the same `labrun_athena_query_optimization_db`. Table name is `events_parquet`. `StorageDescriptor.Location` is `s3://{BUCKET}/curated-parquet/`. Columns are the same as `events_csv` minus the partition columns (the partition columns live in `PartitionKeys`, not the column list, per Hive convention). `PartitionKeys` are `year` (int) and `month` (int).
3. **Configure partition projection on the table.** This is what saves you the crawler. `Parameters` on the table input needs `projection.enabled="true"`, `projection.year.type="integer"`, `projection.year.range="2024,2025"`, `projection.month.type="integer"`, `projection.month.range="1,12"`, and `storage.location.template="s3://{BUCKET}/curated-parquet/year=${year}/month=${month}/"`. Without the template, projection generates partitions but Athena cannot resolve them to S3 paths and you get zero rows.

Heads up on the gotchas: `projection.enabled` must be the string `"true"`, not the Python `True`. The `storage.location.template` substring must contain literal `${year}` and `${month}` (with the dollar-brace syntax); a Python f-string here would silently substitute empty values. Checkpoint 3 verifies all five projection parameters and the template substrings.


In [ ]:
# NBVAL_SKIP
# Task 3: Convert the seeded CSV to partitioned Parquet under
# curated-parquet/year=YYYY/month=MM/, then register events_parquet in the
# Glue Data Catalog with partition projection enabled.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Lazy pyarrow import; Colab ships it preinstalled. Local kernels may need
# pip install pyarrow.
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError:
    print("pyarrow is not installed. Run !pip install pyarrow and re-run this cell.")
    raise


def _convert_csv_to_partitioned_parquet() -> int:
    """Read the seeded CSV from S3 and write one Parquet file per (year, month).

    Returns the total number of Parquet files written. The shape of the seed
    is deterministic so the partition count is always len(MONTHS).
    """
    obj = s3.get_object(Bucket=BUCKET_NAME, Key="raw-csv/events.csv")
    body = obj["Body"].read().decode("utf-8")
    reader = csv.DictReader(io.StringIO(body))
    rows_by_partition: dict[tuple[int, int], list[dict]] = {}
    for row in reader:
        y = int(row["event_year"])
        m = int(row["event_month"])
        rows_by_partition.setdefault((y, m), []).append(row)

    files_written = 0
    for (y, m), rows in rows_by_partition.items():
        table = pa.table({
            "event_id": [r["event_id"] for r in rows],
            "event_day": [int(r["event_day"]) for r in rows],
            "customer_id": [r["customer_id"] for r in rows],
            "amount_cents": [int(r["amount_cents"]) for r in rows],
            "event_type": [r["event_type"] for r in rows],
        })
        buf = io.BytesIO()
        pq.write_table(table, buf, compression="snappy")
        buf.seek(0)
        key = f"curated-parquet/year={y}/month={m}/part-0000.parquet"
        s3.put_object(Bucket=BUCKET_NAME, Key=key, Body=buf.getvalue())
        files_written += 1
    return files_written


print("Converting seeded CSV to partitioned Parquet (this takes a few seconds)...")
files = _convert_csv_to_partitioned_parquet()
print(f"Wrote {files} Parquet file(s) under s3://{BUCKET_NAME}/curated-parquet/")

# events_parquet table input. Partition keys live in PartitionKeys, NOT in
# StorageDescriptor.Columns (Hive convention). The Parameters dict carries
# the partition-projection config; storage.location.template must contain
# literal ${year} and ${month} substrings.
parquet_table_input = {
    "Name": PARQUET_TABLE_NAME,
    "TableType": "EXTERNAL_TABLE",
    "Parameters": {
        "classification": "parquet",
        "has_encrypted_data": "false",
        "projection.enabled": "true",
        "projection.year.type": "integer",
        "projection.year.range": "2024,2025",
        "projection.month.type": "integer",
        "projection.month.range": "1,12",
        "storage.location.template": (
            f"s3://{BUCKET_NAME}/curated-parquet/year=${{year}}/month=${{month}}/"
        ),
    },
    "StorageDescriptor": {
        "Columns": [
            {"Name": "event_id", "Type": "string"},
            {"Name": "event_day", "Type": "int"},
            {"Name": "customer_id", "Type": "string"},
            {"Name": "amount_cents", "Type": "int"},
            {"Name": "event_type", "Type": "string"},
        ],
        "Location": f"s3://{BUCKET_NAME}/curated-parquet/",
        "InputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat",
        "OutputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat",
        "SerdeInfo": {
            "SerializationLibrary": "org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe",
        },
    },
    "PartitionKeys": [
        {"Name": "year", "Type": "int"},
        {"Name": "month", "Type": "int"},
    ],
}

# YOUR CODE: Register the Parquet table with glue.create_table(
#   DatabaseName=DATABASE_NAME,
#   TableInput=parquet_table_input,
# ). Wrap in try/except ClientError and reuse on AlreadyExistsException.

print(f"Parquet table registered: {DATABASE_NAME}.{PARQUET_TABLE_NAME}")
print(f"  Location: s3://{BUCKET_NAME}/curated-parquet/")
print(f"  Partition keys: year (int), month (int)")
print(f"  projection.enabled: true (year and month integer projections)")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: events_parquet exists with partition projection enabled and
# the year/month partition keys. Validator walks Parameters and PartitionKeys.

REQUIRED_PROJECTION_PARAMS = {
    "projection.enabled": "true",
    "projection.year.type": "integer",
    "projection.month.type": "integer",
    "projection.month.range": "1,12",
}


def checkpoint_3(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            table_resp = glue_client.get_table(
                DatabaseName=DATABASE_NAME, Name=PARQUET_TABLE_NAME,
            )
        except ClientError as e:
            if e.response["Error"]["Code"] == "EntityNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Glue table {DATABASE_NAME}.{PARQUET_TABLE_NAME} does not "
                        f"exist. Run the Task 3 cell."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        table = table_resp["Table"]
        sd = table.get("StorageDescriptor", {})
        expected_location = f"s3://{BUCKET_NAME}/curated-parquet/"
        actual_location = sd.get("Location", "")
        if actual_location.rstrip("/") + "/" != expected_location:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"events_parquet StorageDescriptor.Location is "
                    f"{actual_location!r}, expected {expected_location!r}."
                ),
            )

        # PartitionKeys must include both year and month.
        partition_keys = table.get("PartitionKeys", []) or []
        names = {pk.get("Name") for pk in partition_keys}
        for required in ("year", "month"):
            if required not in names:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"events_parquet PartitionKeys is missing {required!r}. "
                        f"PartitionKeys must include both year and month so "
                        f"projection can prune the scan."
                    ),
                )

        # Projection parameters.
        params = table.get("Parameters", {}) or {}
        for key, expected in REQUIRED_PROJECTION_PARAMS.items():
            actual = params.get(key)
            if actual != expected:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Table Parameter {key}={actual!r}, expected {expected!r}. "
                        f"projection.enabled must be the STRING 'true' (lowercase), "
                        f"and the year/month projection types must be 'integer'."
                    ),
                )

        if "projection.year.range" not in params:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Table Parameters is missing projection.year.range. Set it "
                    "to '2024,2025' (or any range that covers the seeded data)."
                ),
            )

        template = params.get("storage.location.template", "")
        if "${year}" not in template or "${month}" not in template:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"storage.location.template is {template!r}; it must "
                    f"contain literal ${{year}} and ${{month}} substrings so "
                    f"Athena can resolve projected partitions to S3 paths. "
                    f"A Python f-string would silently substitute empty values; "
                    f"escape the braces or use a regular string literal."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks five things: `events_parquet` exists in the Glue Data Catalog, its `StorageDescriptor.Location` is exactly `s3://{BUCKET}/curated-parquet/`, `PartitionKeys` includes both `year` and `month`, the five projection parameters match the expected strings (`projection.enabled="true"`, `projection.year.type="integer"`, `projection.month.type="integer"`, `projection.month.range="1,12"`), and `storage.location.template` contains literal `${year}` and `${month}` substrings. Read the failure reason. The two most common misses are (1) using Python `True` instead of the string `"true"` for `projection.enabled` and (2) accidentally f-stringing the template so `${year}` becomes empty.

</details>

<details><summary>Hint 2 (direction)</summary>

The Parquet seed step uses `pyarrow.parquet.write_table` and is already written in the cell. The only API call you add is `glue.create_table(DatabaseName=DATABASE_NAME, TableInput=parquet_table_input)`. Wrap it in `try / except ClientError` and reuse on `AlreadyExistsException` so re-running the cell is idempotent. The `parquet_table_input` dict carries every projection parameter the checkpoint cares about; just pass it through.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3:

```python
try:
    glue.create_table(
        DatabaseName=DATABASE_NAME,
        TableInput=parquet_table_input,
    )
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        pass
    else:
        raise
```

If a query against `events_parquet` returns zero rows where you expect data, `storage.location.template` is wrong. The most common cause is rewriting the template as a Python f-string (`f"s3://...year={year}/month={month}/"`) which substitutes the literal Python variable names; the table-input cell above uses `f"...year=${{year}}/month=${{month}}/"` (double-brace escape) so the dollar-brace literals survive into Glue. If `projection.enabled` is the Python `True` instead of the string `"true"`, Glue silently disables projection and Athena falls back to non-existent registered partitions, also returning zero rows.

</details>


**Wallet check.** Still under one cent total. The Parquet conversion is pure compute in this notebook (free) and a handful of S3 PUT requests (well inside the always-free tier). Registering the second table is another free Glue control-plane call. The bill jumps next cell when you scan Parquet, but only by another fraction of a penny.


## Task 4: Run the same query against `events_parquet` and prove the 80%+ scan reduction

Same logical query, partitioned table:

```sql
SELECT count(*) FROM events_parquet WHERE year = 2025
```

That `WHERE year = 2025` is doing the heavy lifting. With projection enabled, Athena evaluates the predicate against the projected partition values (year in `[2024, 2025]`, month in `[1, 12]`) before touching S3 at all. Only the partitions that satisfy `year = 2025` get listed and scanned. The columnar Parquet format compounds the win: even within the scanned partitions, Athena reads only the column data it needs for `count(*)`, which is roughly nothing (it can use Parquet row group statistics for the count).

Result on the seeded dataset: the Parquet scan is typically 1 to 10 KiB versus the CSV scan of about 350 to 400 KiB. That is a 95%-plus reduction, well past the 80% bar. Checkpoint 4 reads the byte count from this query, compares it against the cached `CSV_SCAN_BYTES`, and passes if `parquet_scan_bytes <= 0.2 * CSV_SCAN_BYTES`.

The classic gotcha here: omit the `WHERE year = ...` clause and Athena scans every partition, which on this dataset is roughly the same byte count as the unfiltered CSV. The hint progression covers this; you will write the query with the predicate.


In [ ]:
# NBVAL_SKIP
# Task 4: Run the same logical query against events_parquet and cache the
# Parquet DataScannedInBytes in PARQUET_SCAN_BYTES for Checkpoint 4.

athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)


def run_athena_query(sql: str, workgroup: str) -> dict:
    """Submit a query, poll until terminal, return the get_query_execution dict."""
    start = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": DATABASE_NAME},
        WorkGroup=workgroup,
    )
    qid = start["QueryExecutionId"]
    deadline = time.time() + 120
    while time.time() < deadline:
        resp = athena.get_query_execution(QueryExecutionId=qid)
        state = resp["QueryExecution"]["Status"]["State"]
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            if state != "SUCCEEDED":
                reason = resp["QueryExecution"]["Status"].get("StateChangeReason", "(no reason)")
                raise RuntimeError(f"Query {qid} ended {state}: {reason}")
            return resp["QueryExecution"]
        time.sleep(2)
    raise RuntimeError(f"Query {qid} did not finish within 2 minutes.")


# YOUR CODE: Write PARQUET_SQL as the same logical query against events_parquet
# with a year = 2025 predicate. The predicate is what triggers partition
# projection pruning. A query without the predicate scans every partition and
# defeats the optimization.
PARQUET_SQL = f"SELECT count(*) FROM {PARQUET_TABLE_NAME} WHERE year = 2025"
print(f"Running Parquet query in {WORKGROUP_NAME}: {PARQUET_SQL}")

parquet_exec = run_athena_query(PARQUET_SQL, WORKGROUP_NAME)
PARQUET_SCAN_BYTES = parquet_exec.get("Statistics", {}).get("DataScannedInBytes", 0)
print(f"Parquet DataScannedInBytes: {PARQUET_SCAN_BYTES} bytes ({PARQUET_SCAN_BYTES / 1024:.2f} KiB)")

if CSV_SCAN_BYTES and CSV_SCAN_BYTES > 0:
    reduction = 1.0 - (PARQUET_SCAN_BYTES / CSV_SCAN_BYTES)
    print(f"CSV baseline: {CSV_SCAN_BYTES} bytes ({CSV_SCAN_BYTES / 1024:.1f} KiB)")
    print(f"Reduction: {reduction * 100:.1f}%")
    if reduction >= 0.8:
        print("PASS-projection target: at least 80% less data scanned.")
    else:
        print("Below the 80% target. Confirm the WHERE clause references the year partition key.")
else:
    print("CSV_SCAN_BYTES is not set. Re-run the Task 2 cell to capture the baseline.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Parquet query scanned at least 80% less data than the CSV
# baseline. Reads CSV_SCAN_BYTES from Checkpoint 2 and PARQUET_SCAN_BYTES
# from Task 4.

REDUCTION_TARGET = 0.8  # 80% less data scanned vs the CSV baseline.


def checkpoint_4(session):
    global PARQUET_SCAN_BYTES
    try:
        if not CSV_SCAN_BYTES or CSV_SCAN_BYTES <= 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "CSV_SCAN_BYTES is not set or is zero. Re-run the Task 2 "
                    "cell so the baseline byte count is captured before "
                    "comparing the Parquet scan."
                ),
            )

        # If the task cell did not run, run the Parquet query here so the
        # checkpoint is independent of cell ordering.
        if PARQUET_SCAN_BYTES is None or PARQUET_SCAN_BYTES == 0:
            athena_client = boto3.client(
                "athena",
                aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
                aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
                aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
                region_name=REGION,
            )
            sql = f"SELECT count(*) FROM {PARQUET_TABLE_NAME} WHERE year = 2025"
            try:
                start = athena_client.start_query_execution(
                    QueryString=sql,
                    QueryExecutionContext={"Database": DATABASE_NAME},
                    WorkGroup=WORKGROUP_NAME,
                )
            except ClientError as e:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Could not start Parquet query in {WORKGROUP_NAME!r}: "
                        f"{e}. Verify Checkpoint 1 still passes."
                    ),
                )
            qid = start["QueryExecutionId"]
            deadline = time.time() + 120
            scanned = None
            while time.time() < deadline:
                resp = athena_client.get_query_execution(QueryExecutionId=qid)
                state = resp["QueryExecution"]["Status"]["State"]
                if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
                    if state != "SUCCEEDED":
                        reason = resp["QueryExecution"]["Status"].get("StateChangeReason", "(no reason)")
                        return CheckpointResult(
                            status="fail",
                            error_reason=(
                                f"Parquet query ended {state}: {reason}. The "
                                f"table is registered but the query could not "
                                f"complete; inspect the table definition."
                            ),
                        )
                    scanned = resp["QueryExecution"].get("Statistics", {}).get("DataScannedInBytes", 0)
                    break
                time.sleep(2)
            if scanned is None:
                return CheckpointResult(
                    status="fail",
                    error_reason="Parquet query did not finish within 2 minutes.",
                )
            PARQUET_SCAN_BYTES = scanned

        threshold = (1.0 - REDUCTION_TARGET) * CSV_SCAN_BYTES
        if PARQUET_SCAN_BYTES > threshold:
            ratio = PARQUET_SCAN_BYTES / CSV_SCAN_BYTES if CSV_SCAN_BYTES else 0
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Parquet scan was {PARQUET_SCAN_BYTES} bytes vs CSV "
                    f"baseline {CSV_SCAN_BYTES} bytes "
                    f"({ratio * 100:.1f}% of baseline). Target is "
                    f"<= {threshold:.0f} bytes (at most 20% of baseline). "
                    f"The usual cause is the WHERE clause not referencing the "
                    f"year partition key, which forces a full scan even on "
                    f"Parquet. Make sure the query is "
                    f"SELECT count(*) FROM {PARQUET_TABLE_NAME} WHERE year = 2025."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint compares `PARQUET_SCAN_BYTES` against `CSV_SCAN_BYTES` and passes if Parquet scanned 80%-plus less. Read the failure reason. The most common miss is the `WHERE` clause not referencing the `year` partition key: without it, Athena scans every projected partition and the byte delta vanishes. The next most common miss is `CSV_SCAN_BYTES` not being set, which means Task 2 did not run; scroll back up and run the baseline query.

</details>

<details><summary>Hint 2 (direction)</summary>

Write `PARQUET_SQL` as `f"SELECT count(*) FROM {PARQUET_TABLE_NAME} WHERE year = 2025"`. The `year = 2025` predicate is the whole optimization: it lets partition projection prune three of the four months at planning time, so Athena never reads Parquet bytes for 2024. The poll loop and byte-capture logic are already in the cell; just set `PARQUET_SQL`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4:

```python
PARQUET_SQL = f"SELECT count(*) FROM {PARQUET_TABLE_NAME} WHERE year = 2025"
```

If the checkpoint fails with Parquet scanning roughly the same as CSV, you probably wrote `SELECT count(*) FROM events_parquet` with no WHERE clause; that forces a full scan. If the checkpoint fails with `CSV_SCAN_BYTES is not set`, the Task 2 cell did not run to completion; re-run Task 2 before Task 4. If the Parquet scan returns zero rows (and zero scanned bytes), `storage.location.template` is wrong in Checkpoint 3's table definition; recheck Task 3 Hint 3.

</details>


**Wallet check.** Still under one cent total for the session. The Parquet query scanned a handful of KiB; a fraction of a fraction of a penny. The takeaway is the ratio, not the absolute. On the real 200 GB CSV scenario from the cert YAML, the same optimization would have dropped each query from about $1 to under 10 cents.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. Lab 05 has no critical (hourly-billed) resources.
#
# labrun-checks 0.3.0 ships AWS adapters for s3_bucket, iam_role,
# glue_database, and glue_table. It does NOT yet support athena_workgroup. A
# _LabAdapter subclass wraps AwsCleanupAdapter and adds athena_workgroup as
# an inline fallback. When the package promotes athena_workgroup in a future
# release, the wrapper can be removed and run_cleanup called against the
# default.

import sys
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter:
    """AwsCleanupAdapter wrapper that adds athena_workgroup support."""

    def __init__(self) -> None:
        self._aws = AwsCleanupAdapter()

    def delete_resource(self, credentials: dict, resource) -> None:
        if resource.type == "athena_workgroup":
            client = boto3.client(
                "athena",
                aws_access_key_id=credentials["aws_access_key_id"],
                aws_secret_access_key=credentials["aws_secret_access_key"],
                aws_session_token=credentials.get("aws_session_token"),
                region_name=credentials.get("region", REGION),
            )
            try:
                client.delete_work_group(
                    WorkGroup=resource.id,
                    RecursiveDeleteOption=True,
                )
            except ClientError as e:
                code = e.response["Error"]["Code"]
                if code in ("InvalidRequestException", "ResourceNotFoundException"):
                    return
                raise
            return
        return self._aws.delete_resource(credentials, resource)

    def describe_resource(self, credentials: dict, resource) -> bool:
        if resource.type == "athena_workgroup":
            client = boto3.client(
                "athena",
                aws_access_key_id=credentials["aws_access_key_id"],
                aws_secret_access_key=credentials["aws_secret_access_key"],
                aws_session_token=credentials.get("aws_session_token"),
                region_name=credentials.get("region", REGION),
            )
            try:
                client.get_work_group(WorkGroup=resource.id)
                return True
            except ClientError as e:
                code = e.response["Error"]["Code"]
                if code in ("InvalidRequestException", "ResourceNotFoundException"):
                    return False
                raise
        return self._aws.describe_resource(credentials, resource)

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        return self._aws.tag_scan(credentials, lab_slug, region)


# Empty the S3 bucket before run_cleanup tears it down. S3 buckets cannot be
# deleted while they contain objects; the bundled s3_bucket adapter only
# deletes one page of objects at a time, and the lab wrote raw CSV plus
# Parquet partitions plus Athena results plus query metadata.
print("Emptying the S3 bucket before teardown...")
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing to cleanup.")

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: under $0.05.** Athena bills per data scanned and you scanned megabytes across the whole session: about 1 MB on the CSV baseline plus a handful of KiB on the Parquet comparison plus a few re-runs of each while debugging. Glue Data Catalog is free at this scale. S3 storage and requests are well inside the always-free tier. The cleanup cell above deleted the workgroup, both tables, the database, and the bucket so the bill stops accruing now. On the real 200 GB CSV table from the cert scenario, the same lab would have cost about $1 per baseline query and 10 cents per Parquet query; the optimization pattern you proved on megabytes translates linearly to gigabytes.


## These are not graded. They are for you.

Three questions worth sitting with before the exam.

1. The workgroup `BytesScannedCutoffPerQuery` you set in Checkpoint 1 cancels a query that crosses the threshold. Walk through the alternatives: a per-query cost alert via CloudWatch Billing, a Lake Formation row-level filter, an IAM `Condition` on the Athena workgroup, or a Service Quotas request for a lower default. For each, identify one DEA-C01 scenario where it is the right tool and one where the workgroup cap is still the better answer. The cap is the only mechanism that stops the query before it runs to completion; the rest detect or alert after the fact.

2. Partition projection trades crawler runs and `MSCK REPAIR TABLE` calls for a static configuration on the table. The trade-off bites when the projection range falls behind reality (for example, you ship the table with `projection.year.range="2024,2025"` and the data lands in 2026). Walk through how you would detect that drift in production: CloudWatch metrics on `DataScannedInBytes` per query, a daily sentinel query that filters on the latest expected partition, or a Glue job that re-derives the range nightly and pushes a `glue.update_table` call. The exam likes scenarios where projection is wrong; recognize the symptom (zero rows on a filtered query against new data).

3. Athena charges $5 per TB scanned and Parquet plus partitioning plus projection knocked the scan down 95%-plus on this lab. The math holds at scale: a daily 200 GB CSV query for an analyst team is roughly $30 per month; the same query on partitioned Parquet with projection is roughly $1.50. Now consider the storage cost side. Re-encoding 200 GB of CSV as Parquet typically lands at 30 to 50 GB on disk thanks to columnar compression, so storage drops too. Identify two cases where you would deliberately keep the raw CSV alongside the optimized Parquet (reproducibility, regulatory) and two cases where you would delete the CSV (cost-sensitive non-regulated, vendor lock-in concern). DEA-C01 expects you to reason about both sides.

## Exam-Style Practice

**Q1.** A data analyst's Athena query against an unpartitioned CSV table scans 200 GB on every run and costs about $1 per query. The team wants to enforce a guardrail so an unfiltered scan can never cost more than 5 cents going forward, without changing the table layout in the next hour. Which is the most direct control?

A) Lower the AWS Service Quotas value for Athena `Active DML queries per workgroup` to 1.

B) Configure a CloudWatch billing alarm at $0.10 per query on the analyst's workgroup.

C) Set `BytesScannedCutoffPerQuery=10737418240` (10 GiB) on the analyst's Athena workgroup so any query crossing the cutoff is cancelled.

D) Apply a Lake Formation row-level filter scoped to a small date range so the analyst can only see a subset of rows.

<details><summary>Show answer</summary>

**C**.

- A) Wrong. Service Quotas controls concurrency, not bytes scanned. Lowering the active-query quota slows the team down without preventing a single 200 GB scan.
- B) Wrong. CloudWatch billing alarms fire after the fact, after the bytes have been scanned and the charge has accrued. The question asks to prevent the cost, not detect it.
- C) Right. `BytesScannedCutoffPerQuery` is the only Athena-native mechanism that cancels a query before it finishes if it crosses a bytes-scanned threshold. At $5 per TB, 10 GiB scanned is about 5 cents. Set on the workgroup, it applies to every query the analyst runs without any table-layout change.
- D) Wrong. A Lake Formation filter restricts visible rows but Athena still scans all the underlying CSV bytes; the cost stays at $1 per query because the scan happens before the filter applies.

</details>

**Q2.** A team registers a Glue table over partitioned Parquet at `s3://lake/orders/year=YYYY/month=MM/` and configures partition projection with `projection.enabled="true"`, `projection.year.type="integer"`, `projection.year.range="2024,2025"`, `projection.month.type="integer"`, and `projection.month.range="1,12"`. A query `SELECT count(*) FROM orders WHERE year = 2025 AND month = 6` returns zero rows even though objects exist under `s3://lake/orders/year=2025/month=6/`. What is the most likely cause?

A) Athena does not support integer projection on the partition keys; switch the types to `enum`.

B) The table is missing the `storage.location.template` parameter, so Athena projects partition values but cannot resolve them to S3 paths.

C) `BytesScannedCutoffPerQuery` on the workgroup is cancelling the query before it returns rows.

D) The Parquet objects are missing `_SUCCESS` markers under each partition, so Athena treats the partitions as empty.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Integer projection is fully supported for partition keys derived from numeric values; `enum` is for fixed string sets. Switching types would not change anything here.
- B) Right. Partition projection has two pieces: the projected values (defined by `projection.*` parameters) and the path template (defined by `storage.location.template`). Without the template, Athena generates the projected partition keys but cannot map them to S3 paths, so it scans nothing and returns zero rows. The template typically looks like `s3://lake/orders/year=${year}/month=${month}/` with literal `${}` placeholders.
- C) Wrong. A workgroup cutoff would surface as a cancelled query with a `BytesScannedCutoff` error, not a successful query returning zero rows.
- D) Wrong. Athena and Parquet do not require `_SUCCESS` markers (that is a Hadoop-style convention used by some Spark writers). Partitions are detected via the S3 listing of the partition prefix, not a marker file.

</details>

**Q3.** A data engineer needs to optimize an Athena workload that queries a 500 GB CSV table partitioned by `event_date` (string, format `YYYY-MM-DD`). The analyst runs five queries per day, each filtered by a 30-day date range, and the bill is $25 per day. Which combination of changes will reduce the cost the most for this access pattern?

A) Switch the table to Parquet without changing partition keys; add partition projection on `event_date`.

B) Switch the table to Parquet, repartition by `year` and `month` (integers), and add partition projection on both new partition keys.

C) Leave the table as CSV but add Athena query result reuse (CTAS into a results bucket) so repeated queries hit the cache.

D) Switch the table to Parquet, partition by `customer_id` (the most common column in WHERE clauses across the team), and skip projection.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Parquet alone helps with columnar pruning but the existing daily partition on `event_date` is too granular for a 30-day range (Athena must list 30 partitions per query), and the integer-vs-string partition type affects projection-range size. Repartitioning to year and month, then projecting both, is the canonical pattern.
- B) Right. The DEA-C01-canonical optimization stack: columnar format (Parquet) for column pruning, coarser partition grain (year and month) for cheaper partition listing on multi-month ranges, and projection on both keys to skip the per-query partition discovery roundtrip. This compounds: most analyst queries land in one to two month partitions, scanning a few GB instead of 500 GB.
- C) Wrong. Result reuse only helps for the exact same query string within the cache window. Five queries per day with different 30-day ranges share little; the savings are minimal compared to changing format and partition layout.
- D) Wrong. Partitioning by `customer_id` creates one S3 prefix per customer (potentially millions), which Athena handles poorly: partition listing dominates query latency and there is a 20,000 partition request limit per query. High-cardinality partition keys are an anti-pattern on the cert. Year/month at the partition level plus `customer_id` in the WHERE clause is the right shape.

</details>
